In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

app = pd.read_csv('../data/raw/application_train.csv')
bureau = pd.read_csv('../data/raw/bureau.csv')

print(f"application_train: {app.shape}")
print(f"bureau: {bureau.shape}")
print(f"\nКолонки bureau:")
print(bureau.columns.tolist())

application_train: (307511, 122)
bureau: (1716428, 17)

Колонки bureau:
['SK_ID_CURR', 'SK_ID_BUREAU', 'CREDIT_ACTIVE', 'CREDIT_CURRENCY', 'DAYS_CREDIT', 'CREDIT_DAY_OVERDUE', 'DAYS_CREDIT_ENDDATE', 'DAYS_ENDDATE_FACT', 'AMT_CREDIT_MAX_OVERDUE', 'CNT_CREDIT_PROLONG', 'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_SUM_LIMIT', 'AMT_CREDIT_SUM_OVERDUE', 'CREDIT_TYPE', 'DAYS_CREDIT_UPDATE', 'AMT_ANNUITY']


In [2]:
credits_per_client = bureau.groupby('SK_ID_CURR').size()

print(f"Среднее кредитов на клиента: {credits_per_client.mean():.1f}")
print(f"Максимум кредитов у одного клиента: {credits_per_client.max()}")
print(f"Медиана: {credits_per_client.median():.0f}")

Среднее кредитов на клиента: 5.6
Максимум кредитов у одного клиента: 116
Медиана: 4


In [ ]:
bureau_agg = bureau.groupby('SK_ID_CURR').agg(
    # Количество кредитов — чем больше, тем опытнее или тем больше долгов
    BUREAU_CREDIT_COUNT=('SK_ID_BUREAU', 'count'),
    
    # Доля активных кредитов — много активных = высокая нагрузка
    BUREAU_ACTIVE_CREDIT_COUNT=('CREDIT_ACTIVE', lambda x: (x == 'Active').sum()),
    
    # Средний текущий долг — общая долговая нагрузка
    BUREAU_AMT_DEBT_MEAN=('AMT_CREDIT_SUM_DEBT', 'mean'),
    
    # Максимальная просроченная сумма — был ли клиент должником
    BUREAU_AMT_OVERDUE_MAX=('AMT_CREDIT_SUM_OVERDUE', 'max'),
    
    # Среднее дней просрочки — насколько часто опаздывал с платежами  
    BUREAU_DAYS_OVERDUE_MEAN=('CREDIT_DAY_OVERDUE', 'mean'),
    
    # Максимальный кредит — масштаб финансовых операций клиента
    BUREAU_AMT_CREDIT_MAX=('AMT_CREDIT_SUM', 'max'),
).reset_index()

print(f"Размер после агрегации: {bureau_agg.shape}")
print(bureau_agg.head())

Размер после агрегации: (305811, 7)
   SK_ID_CURR  BUREAU_CREDIT_COUNT  BUREAU_ACTIVE_CREDIT_COUNT  \
0      100001                    7                           3   
1      100002                    8                           2   
2      100003                    4                           1   
3      100004                    2                           0   
4      100005                    3                           2   

   BUREAU_AMT_DEBT_MEAN  BUREAU_AMT_OVERDUE_MAX  BUREAU_DAYS_OVERDUE_MEAN  \
0          85240.928571                     0.0                       0.0   
1          49156.200000                     0.0                       0.0   
2              0.000000                     0.0                       0.0   
3              0.000000                     0.0                       0.0   
4         189469.500000                     0.0                       0.0   

   BUREAU_AMT_CREDIT_MAX  
0               378000.0  
1               450000.0  
2               810000.

In [ ]:
app = app.merge(bureau_agg, on='SK_ID_CURR', how='left')

print(f"Размер после джойна: {app.shape}")
print(f"Пропуски в новых колонках:")
print(app[bureau_agg.columns[1:]].isnull().mean().apply(lambda x: f"{x:.1%}"))

Размер после джойна: (307511, 128)
Пропуски в новых колонках:
BUREAU_CREDIT_COUNT           14.3%
BUREAU_ACTIVE_CREDIT_COUNT    14.3%
BUREAU_AMT_DEBT_MEAN          16.7%
BUREAU_AMT_OVERDUE_MAX        14.3%
BUREAU_DAYS_OVERDUE_MEAN      14.3%
BUREAU_AMT_CREDIT_MAX         14.3%
dtype: str


In [ ]:
# Возраст в годах
app['AGE_YEARS'] = -app['DAYS_BIRTH'] / 365

# Стаж в годах
# убираем аномалию
app['DAYS_EMPLOYED'] = app['DAYS_EMPLOYED'].replace(365243, np.nan)
app['YEARS_EMPLOYED'] = -app['DAYS_EMPLOYED'] / 365

# Отношение аннуитета к доходу — какую долю дохода съедает платёж
# Высокое значение = клиент перегружен платежами
app['ANNUITY_INCOME_RATIO'] = app['AMT_ANNUITY'] / app['AMT_INCOME_TOTAL']

# Отношение кредита к доходу — масштаб кредита относительно возможностей
app['CREDIT_INCOME_RATIO'] = app['AMT_CREDIT'] / app['AMT_INCOME_TOTAL']

# Отношение кредита к стоимости товара
# Если кредит >> стоимости товара — клиент берёт больше чем нужно
app['CREDIT_GOODS_RATIO'] = app['AMT_CREDIT'] / app['AMT_GOODS_PRICE']

# Среднее внешних скорингов
app['EXT_SOURCE_MEAN'] = app[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].mean(axis=1)

# Произведение внешних скорингов — когда все три низкие
app['EXT_SOURCE_PROD'] = (app['EXT_SOURCE_1'] * 
                          app['EXT_SOURCE_2'] * 
                          app['EXT_SOURCE_3'])

print(app[['AGE_YEARS', 'ANNUITY_INCOME_RATIO', 'CREDIT_INCOME_RATIO', 
           'EXT_SOURCE_MEAN', 'EXT_SOURCE_PROD']].describe().round(2))
app = app.copy()

Новые фичи созданы
       AGE_YEARS  ANNUITY_INCOME_RATIO  CREDIT_INCOME_RATIO  EXT_SOURCE_MEAN  \
count  307511.00             307499.00            307511.00        307339.00   
mean       43.94                  0.18                 3.96             0.51   
std        11.96                  0.09                 2.69             0.15   
min        20.52                  0.00                 0.00             0.00   
25%        34.01                  0.11                 2.02             0.41   
50%        43.15                  0.16                 3.27             0.52   
75%        53.92                  0.23                 5.16             0.62   
max        69.12                  1.88                84.74             0.88   

       EXT_SOURCE_PROD  
count        109589.00  
mean              0.14  
std               0.11  
min               0.00  
25%               0.06  
50%               0.12  
75%               0.21  
max               0.62  
Датафрейм дефрагментирован
Итоговый

In [ ]:
from sklearn.preprocessing import LabelEncoder

cat_cols = app.select_dtypes(include='object').columns.tolist()
print(f"Категориальных колонок: {len(cat_cols)}")
print(cat_cols)

le = LabelEncoder()
for col in cat_cols:
    app[col] = le.fit_transform(app[col].fillna('Unknown'))

print(app[cat_cols].head())

C:\Users\Paul\AppData\Local\Temp\ipykernel_22924\3691688500.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = app.select_dtypes(include='object').columns.tolist()


Категориальных колонок: 16
['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']

После кодирования:
   NAME_CONTRACT_TYPE  CODE_GENDER  FLAG_OWN_CAR  FLAG_OWN_REALTY  \
0                   0            1             0                1   
1                   0            0             0                0   
2                   1            1             1                1   
3                   0            0             0                1   
4                   0            1             0                1   

   NAME_TYPE_SUITE  NAME_INCOME_TYPE  NAME_EDUCATION_TYPE  NAME_FAMILY_STATUS  \
0                6                 7                    4                   3   
1                1                 4       

In [ ]:
feature_cols = [
    # Оригинальные топ фичи из EDA
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_INCOME_TOTAL',
    'DAYS_BIRTH', 'DAYS_EMPLOYED',
    'AMT_GOODS_PRICE', 'REGION_POPULATION_RELATIVE',
    
    # Новые фичи из application_train
    'AGE_YEARS', 'YEARS_EMPLOYED',
    'ANNUITY_INCOME_RATIO', 'CREDIT_INCOME_RATIO',
    'CREDIT_GOODS_RATIO', 'EXT_SOURCE_MEAN', 'EXT_SOURCE_PROD',
    
    # Категориальные (закодированные)
    'CODE_GENDER', 'NAME_CONTRACT_TYPE', 'NAME_EDUCATION_TYPE',
    'NAME_INCOME_TYPE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE',
    
    # Фичи из bureau
    'BUREAU_CREDIT_COUNT', 'BUREAU_ACTIVE_CREDIT_COUNT',
    'BUREAU_AMT_DEBT_MEAN', 'BUREAU_AMT_OVERDUE_MAX',
    'BUREAU_DAYS_OVERDUE_MEAN', 'BUREAU_AMT_CREDIT_MAX'
]

X = app[feature_cols]
y = app['TARGET']

pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

scores = cross_val_score(pipeline, X, y, cv=5, scoring='roc_auc', n_jobs=-1)

print(f"Baseline ROC-AUC:       0.7258")
print(f"После feature eng:      {scores.mean():.4f} ± {scores.std():.4f}")
print(f"Прирост:                +{scores.mean() - 0.7258:.4f}")

Baseline ROC-AUC:       0.7258
После feature eng:      0.7407 ± 0.0023
Прирост:                +0.0149
